In [ ]:
%pip install timm==0.9.12 torch torchmetrics torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade tqdm opencv-python pillow --upgrade

In [2]:
import torch
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0)) 

GPU name: NVIDIA GeForce RTX 4060 Ti


In [2]:
# Cell 1
from pathlib import Path
import re, hashlib, cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
from tqdm import tqdm
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

# ⚙️ SET YOUR DATA ROOT
DATA_ROOT = Path(r"G:/My Drive/CLPD-MF-Dataset")  
assert DATA_ROOT.exists(), f"Dataset folder not found at {DATA_ROOT}"
print("Found dataset root:", DATA_ROOT)


Found dataset root: G:\My Drive\CLPD-MF-Dataset


In [6]:
def list_images_and_labels(root):
    """
    List all images with labels and magnifications from the new folder structure.
    
    Structure:
    G:/My Drive/CLPD-MF-Dataset/
    ├── MF/
    │   └── Patient Name/
    │       ├── x5/
    │       ├── x10/
    │       └── x20/
    └── Non-MF/
        ├── B cell Lymphoma/
        │   └── Patient Name/
        │       ├── x5/
        │       ├── x10/
        │       └── x20/
        ├── PLEVA-PLC/
        └── pseudolymphoma/
    """
    rows = []
    root = Path(root)
    
    # Process MF folder
    mf_dir = root / "MF"
    if mf_dir.exists():
        for patient_dir in mf_dir.iterdir():
            if not patient_dir.is_dir(): continue
            
            patient_name = patient_dir.name
            # Look for x10 and x20 subfolders
            for mag in ['x10', 'x20']:
                mag_dir = patient_dir / mag
                if mag_dir.exists() and mag_dir.is_dir():
                    # Find all .tif images in this magnification folder
                    for img_path in mag_dir.glob('*.tif'):
                        rows.append({
                            'path': img_path,
                            'label': 'MF',
                            'patient': patient_name,
                            'mag': mag,
                            'subtype': None  # MF has no subtype
                        })
    
    # Process Non-MF folder with subtypes
    nonmf_dir = root / "Non-MF"
    if nonmf_dir.exists():
        # Each subfolder is a disease subtype
        for subtype_dir in nonmf_dir.iterdir():
            if not subtype_dir.is_dir(): continue
            
            subtype = subtype_dir.name  # B cell Lymphoma, PLEVA-PLC, or pseudolymphoma
            
            # Each patient within the subtype
            for patient_dir in subtype_dir.iterdir():
                if not patient_dir.is_dir(): continue
                
                patient_name = patient_dir.name
                
                # Look for x10 and x20 subfolders
                for mag in ['x10', 'x20']:
                    mag_dir = patient_dir / mag
                    if mag_dir.exists() and mag_dir.is_dir():
                        # Find all .tif images in this magnification folder
                        for img_path in mag_dir.glob('*.tif'):
                            rows.append({
                                'path': img_path,
                                'label': 'Non-MF',
                                'patient': patient_name,
                                'mag': mag,
                                'subtype': subtype
                            })
    
    return rows

# Load all images
print("\n" + "="*80)
print("LOADING DATASET")
print("="*80 + "\n")

all_images = list_images_and_labels(DATA_ROOT)

print(f"Total images found: {len(all_images)}")
print(f"\nMagnification distribution:")
mag_counts = Counter([r['mag'] for r in all_images])
for mag, count in sorted(mag_counts.items()):
    print(f"  {mag}: {count} images")

print(f"\nLabel distribution:")
label_counts = Counter([r['label'] for r in all_images])
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count} images")

# Count unique patients
unique_patients = len(set(r['patient'] for r in all_images))
mf_patients = len(set(r['patient'] for r in all_images if r['label'] == 'MF'))
nonmf_patients = len(set(r['patient'] for r in all_images if r['label'] == 'Non-MF'))
print(f"\nUnique patients:")
print(f"  Total: {unique_patients}")
print(f"  MF: {mf_patients}")
print(f"  Non-MF: {nonmf_patients}")

# Show Non-MF subtypes distribution
print(f"\nNon-MF subtypes:")
nonmf_images = [r for r in all_images if r['label'] == 'Non-MF']
subtype_counts = Counter([r['subtype'] for r in nonmf_images])
for subtype, count in sorted(subtype_counts.items()):
    subtype_patients = len(set(r['patient'] for r in nonmf_images if r['subtype'] == subtype))
    print(f"  {subtype}: {count} images ({subtype_patients} patients)")




LOADING DATASET

Total images found: 1106

Magnification distribution:
  x10: 413 images
  x20: 693 images

Label distribution:
  MF: 460 images
  Non-MF: 646 images

Unique patients:
  Total: 63
  MF: 21
  Non-MF: 42

Non-MF subtypes:
  B cell Lymphoma: 291 images (16 patients)
  PLEVA-PLC: 209 images (16 patients)
  pseudolymphoma: 146 images (10 patients)


In [7]:
# Cell 3
PATCH_CACHE = Path('./patch_cache')
PATCH_CACHE.mkdir(exist_ok=True)

def extract_and_cache_patches(img_path, patch_size=512, stride=256, 
                              min_foreground_ratio=0.05, max_patches_per_image=200):
    key = hashlib.sha1(str(img_path).encode()).hexdigest()
    cache_dir = PATCH_CACHE / key
    if cache_dir.exists() and any(cache_dir.iterdir()):
        return sorted([str(p) for p in cache_dir.glob('*.jpg')])

    cache_dir.mkdir(parents=True, exist_ok=True)
    img = Image.open(img_path).convert('RGB')
    W,H = img.size
    patches = []

    for y in range(0, H-patch_size+1, stride):
        for x in range(0, W-patch_size+1, stride):
            crop = img.crop((x,y,x+patch_size,y+patch_size))
            arr = np.asarray(crop)
            v = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            fg_ratio = (v > 10).mean()
            if fg_ratio < min_foreground_ratio: continue
            fname = cache_dir / f'{x}_{y}.jpg'
            crop.save(fname, quality=90)
            patches.append(str(fname))
            if len(patches) >= max_patches_per_image: break
        if len(patches) >= max_patches_per_image: break
    return patches


In [ ]:
# Cell 4
class MFHistologyDataset(Dataset):
    def __init__(self, rows, mag='x20', mode='train', patching=True, patch_size=512,
                 stride=256, transforms=None, max_patches_per_image=100):
        self.rows = [r for r in rows if (mag is None or r['mag']==mag)]
        self.mode = mode
        self.patching = patching
        self.patch_size = patch_size
        self.stride = stride
        self.max_patches_per_image = max_patches_per_image
        self.transforms = transforms
        labels = sorted(list({r['label'] for r in self.rows}))
        self.label2idx = {lab:i for i,lab in enumerate(labels)}


        self.items = []
        for r in self.rows:
            if self.patching:
                patches = extract_and_cache_patches(r['path'], patch_size=self.patch_size,
                                                    stride=self.stride, max_patches_per_image=self.max_patches_per_image)
                for p in patches:
                    self.items.append({'img': p, 'label': self.label2idx[r['label']], 'source': str(r['path'])})
            else:
                self.items.append({'img': str(r['path']), 'label': self.label2idx[r['label']], 'source': str(r['path'])})
        if len(self.items)==0:
            print("Warning: dataset empty for magnification", mag)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        img = Image.open(it['img']).convert('RGB')
        if self.transforms: img = self.transforms(img)
        return img, it['label'], it['source']


train_tf = transforms.Compose([
    transforms.Resize((512,512), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((512,512), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


In [ ]:
# Cell 5
def get_patient_level_folds(rows, mag='x20', n_splits=5, seed=42):
    """
    Creates patient-level stratified K-fold splits.
    Returns list of (train_rows, val_rows) tuples for each fold.
    """
    # Group patients by class
    cls_map = {}
    for r in rows:
        if r['mag'] != mag:
            continue
        cls_map.setdefault(r['label'], {}).setdefault(r['patient'], []).append(r)
    
    # Get unique patients per class
    patient_data = {'MF': [], 'Non-MF': []}
    for cls, patients_dict in cls_map.items():
        for patient, patient_rows in patients_dict.items():
            patient_data[cls].append({
                'patient': patient,
                'rows': patient_rows,
                'label': cls
            })
    
    # Create patient arrays and labels for stratification
    all_patient_info = []
    for cls in patient_data:
        all_patient_info.extend(patient_data[cls])
    
    # Extract patient IDs and labels for sklearn
    patient_ids = [p['patient'] for p in all_patient_info]
    patient_labels = [p['label'] for p in all_patient_info]
    
    # Create stratified K-fold splitter
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    # Generate folds
    folds = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(patient_ids, patient_labels)):
        train_patients = set([all_patient_info[i]['patient'] for i in train_idx])
        val_patients = set([all_patient_info[i]['patient'] for i in val_idx])
        
        # Collect all rows for train/val patients
        train_rows = []
        val_rows = []
        for info in all_patient_info:
            if info['patient'] in train_patients:
                train_rows.extend(info['rows'])
            elif info['patient'] in val_patients:
                val_rows.extend(info['rows'])
        
        folds.append({
            'fold': fold_idx + 1,
            'train_rows': train_rows,
            'val_rows': val_rows,
            'train_patients': train_patients,
            'val_patients': val_patients
        })
        
        # Print fold info
        train_mf = sum(1 for r in train_rows if r['label'] == 'MF')
        val_mf = sum(1 for r in val_rows if r['label'] == 'MF')
        print(f"Fold {fold_idx+1}: Train patients={len(train_patients)} ({train_mf} MF, {len(train_rows)-train_mf} Non-MF images), "
              f"Val patients={len(val_patients)} ({val_mf} MF, {len(val_rows)-val_mf} Non-MF images)")
    
    return folds

# Generate folds for both magnifications
print("=" * 60)
print("Creating 5-fold CV splits for x20...")
print("=" * 60)
folds_20 = get_patient_level_folds(all_images, mag='x20', n_splits=5, seed=42)

print("\n" + "=" * 60)
print("Creating 5-fold CV splits for x10...")
print("=" * 60)
folds_10 = get_patient_level_folds(all_images, mag='x10', n_splits=5, seed=42)

Creating 5-fold CV splits for x20...
Fold 1: Train patients=28 (297 MF, 142 Non-MF images), Val patients=8 (58 MF, 34 Non-MF images)
Fold 2: Train patients=29 (279 MF, 156 Non-MF images), Val patients=7 (76 MF, 20 Non-MF images)
Fold 3: Train patients=29 (240 MF, 121 Non-MF images), Val patients=7 (115 MF, 55 Non-MF images)
Fold 4: Train patients=29 (288 MF, 140 Non-MF images), Val patients=7 (67 MF, 36 Non-MF images)
Fold 5: Train patients=29 (316 MF, 145 Non-MF images), Val patients=7 (39 MF, 31 Non-MF images)

Creating 5-fold CV splits for x10...
Fold 1: Train patients=23 (84 MF, 93 Non-MF images), Val patients=6 (16 MF, 26 Non-MF images)
Fold 2: Train patients=23 (89 MF, 92 Non-MF images), Val patients=6 (11 MF, 27 Non-MF images)
Fold 3: Train patients=23 (68 MF, 93 Non-MF images), Val patients=6 (32 MF, 26 Non-MF images)
Fold 4: Train patients=23 (80 MF, 96 Non-MF images), Val patients=6 (20 MF, 23 Non-MF images)
Fold 5: Train patients=24 (79 MF, 102 Non-MF images), Val patients=5

In [3]:
# Cell 6 - Model and Utility Functions for K-Fold CV
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast

# Set device
device = torch.device('cuda')

# Model creation function
def create_model(model_name='resnet50', pretrained=True, num_classes=2, dropout=0.2):
    """
    Creates a timm model with specified architecture.
    
    Available models:
    - 'resnet50': ResNet50
    - 'tf_efficientnet_b2': EfficientNet-B2
    - 'swin_base_patch4_window7_224': Swin Transformer Base
    """
    model = timm.create_model(
        model_name, 
        pretrained=pretrained, 
        num_classes=num_classes, 
        drop_rate=dropout
    )
    return model

# Class weights computation function
def get_class_weights(train_ds):
    """
    Computes balanced class weights for handling class imbalance.
    """
    counts = {}
    for it in train_ds.items:
        counts[it['label']] = counts.get(it['label'], 0) + 1
    
    total = sum(counts.values())
    weights = [total / counts.get(i, 1) for i in range(len(counts))]
    
    print(f"  Class distribution: {counts}")
    print(f"  Class weights: {[f'{w:.4f}' for w in weights]}")
    
    return torch.tensor(weights, dtype=torch.float).to(device)

# Model configurations for each magnification
# You can experiment with different architectures here
model_configs = {
    'x10': {
        'model_name': 'tf_efficientnet_b2',
        'lr': 3e-4,
        'dropout': 0.3,
        'epochs': 5,
        'batch_size': 8
    },
    'x20': {
        'model_name': 'tf_efficientnet_b2',
        'lr': 3e-4,
        'dropout': 0.3,
        'epochs': 5,
        'batch_size': 8
    }
}

print("\nModel Configurations:")
print("=" * 60)
for mag, config in model_configs.items():
    print(f"{mag}:")
    for key, value in config.items():
        print(f"  {key}: {value}")
    print()

# Test model creation (optional - just to verify it works)
print("Testing model creation...")
test_model = create_model(
    model_name=model_configs['x20']['model_name'], 
    pretrained=True, 
    num_classes=2,
    dropout=model_configs['x20']['dropout']
)
print(f"✓ Successfully created {model_configs['x20']['model_name']}")
print(f"  Total parameters: {sum(p.numel() for p in test_model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in test_model.parameters() if p.requires_grad):,}")
del test_model  # Free memory
torch.cuda.empty_cache()

c:\Users\Mohamed Hazem\anaconda3\envs\dlclass\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Model Configurations:
x10:
  model_name: tf_efficientnet_b2
  lr: 0.0003
  dropout: 0.3
  epochs: 5
  batch_size: 8

x20:
  model_name: tf_efficientnet_b2
  lr: 0.0003
  dropout: 0.3
  epochs: 5
  batch_size: 8

Testing model creation...
✓ Successfully created tf_efficientnet_b2
  Total parameters: 7,703,812
  Trainable parameters: 7,703,812


In [ ]:
# Cell 7 - K-Fold CV Training with Comprehensive Evaluation
import time
from pathlib import Path
import json
from sklearn.metrics import (
    precision_recall_fscore_support, 
    confusion_matrix, 
    classification_report,
    roc_auc_score
)

save_path = Path("G:/My Drive/CLPD-MF-Dataset/Local Models/CV_Results")
save_path.mkdir(parents=True, exist_ok=True)

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0
    
    for imgs, labels, _src in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    
    return running_loss/total, correct/total

def validate(model, loader, criterion, device):
    """Validate and return comprehensive metrics"""
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            # Get probabilities for AUC calculation
            probs = torch.softmax(outputs, dim=1)
            
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
            
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())
    
    acc = correct/total if total > 0 else 0
    loss = running_loss/total if total > 0 else 0
    
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    all_probs = torch.cat(all_probs)
    
    return loss, acc, all_preds, all_labels, all_probs

def compute_metrics(labels, preds, probs, class_names=['MF', 'Non-MF']):
    """Compute comprehensive evaluation metrics"""
    # Basic metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        labels.numpy(), preds.numpy(), average=None, zero_division=0
    )
    
    # Weighted averages
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        labels.numpy(), preds.numpy(), average='weighted', zero_division=0
    )
    
    # Confusion matrix
    cm = confusion_matrix(labels.numpy(), preds.numpy())
    
    # AUC (handle case where only one class is present)
    try:
        if len(np.unique(labels.numpy())) > 1:
            auc = roc_auc_score(labels.numpy(), probs.numpy()[:, 1])
        else:
            auc = None
    except:
        auc = None
    
    metrics = {
        'precision_per_class': precision.tolist(),
        'recall_per_class': recall.tolist(),
        'f1_per_class': f1.tolist(),
        'support_per_class': support.tolist(),
        'precision_weighted': float(precision_w),
        'recall_weighted': float(recall_w),
        'f1_weighted': float(f1_w),
        'confusion_matrix': cm.tolist(),
        'auc': float(auc) if auc is not None else None,
        'class_names': class_names
    }
    
    return metrics

def train_kfold_cv(folds, mag_name, model_name, epochs=10, batch_size=8, lr=1e-4, dropout=0.3):
    """
    Train model using K-fold cross-validation with comprehensive evaluation.
    """
    results = []
    
    print(f"\n{'='*80}")
    print(f"Starting {len(folds)}-Fold Cross-Validation for {mag_name}")
    print(f"Model: {model_name} | Epochs: {epochs} | Batch Size: {batch_size} | LR: {lr}")
    print(f"{'='*80}\n")
    
    for fold_data in folds:
        fold_idx = fold_data['fold']
        print(f"\n{'='*60}")
        print(f"Fold {fold_idx}/{len(folds)} - {mag_name}")
        print(f"{'='*60}")
        
        # Create datasets for this fold
        train_ds = MFHistologyDataset(
            fold_data['train_rows'], 
            mag=mag_name, 
            mode='train', 
            patching=True, 
            transforms=train_tf,
            max_patches_per_image=100
        )
        val_ds = MFHistologyDataset(
            fold_data['val_rows'], 
            mag=mag_name, 
            mode='val', 
            patching=True, 
            transforms=val_tf,
            max_patches_per_image=100
        )
        
        print(f"Train patches: {len(train_ds)}, Val patches: {len(val_ds)}")
        print(f"Train patients: {len(fold_data['train_patients'])}, Val patients: {len(fold_data['val_patients'])}")
        
        # Create dataloaders
        train_loader = DataLoader(
            train_ds, 
            batch_size=batch_size, 
            shuffle=True, 
            num_workers=0, 
            pin_memory=True
        )
        val_loader = DataLoader(
            val_ds, 
            batch_size=batch_size, 
            shuffle=False, 
            num_workers=0, 
            pin_memory=True
        )
        
        # Create model
        model = create_model(
            model_name=model_name, 
            pretrained=True, 
            num_classes=2,
            dropout=dropout
        ).to(device)
        
        # Class weights for imbalanced data
        weights = get_class_weights(train_ds)
        criterion = nn.CrossEntropyLoss(weight=weights)
        
        # Optimizer with weight decay for regularization
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        
        # Track best model for this fold
        best_val_f1 = 0.0
        best_epoch = 0
        fold_history = []
        patience = 3
        patience_counter = 0
        
        # Training loop
        for epoch in range(epochs):
            t0 = time.time()
            
            train_loss, train_acc = train_one_epoch(
                model, train_loader, optimizer, criterion, device, scaler
            )
            
            val_loss, val_acc, val_preds, val_labels, val_probs = validate(
                model, val_loader, criterion, device
            )
            
            # Compute detailed metrics
            val_metrics = compute_metrics(val_labels, val_preds, val_probs)
            val_f1 = val_metrics['f1_weighted']
            
            t1 = time.time()
            
            print(f"Epoch {epoch+1}/{epochs} | "
                  f"train_loss {train_loss:.4f} train_acc {train_acc:.4f} | "
                  f"val_loss {val_loss:.4f} val_acc {val_acc:.4f} val_f1 {val_f1:.4f} | "
                  f"time {(t1-t0):.1f}s")
            
            fold_history.append({
                'epoch': epoch + 1,
                'train_loss': train_loss,
                'train_acc': train_acc,
                'val_loss': val_loss,
                'val_acc': val_acc,
                'val_f1': val_f1
            })
            
            # Save best model based on F1-score (better for imbalanced data)
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_epoch = epoch + 1
                torch.save(
                    model.state_dict(), 
                    save_path / f'model_{model_name}_{mag_name}_fold{fold_idx}.pth'
                )
                print(f"  → Saved best model (val_f1: {val_f1:.4f})")
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Early stopping
            if patience_counter >= patience:
                print(f"  → Early stopping triggered at epoch {epoch+1}")
                break
        
        # Load best model and compute final metrics
        print(f"\nLoading best model from epoch {best_epoch}...")
        model.load_state_dict(
            torch.load(save_path / f'model_{model_name}_{mag_name}_fold{fold_idx}.pth')
        )
        
        val_loss, val_acc, val_preds, val_labels, val_probs = validate(
            model, val_loader, criterion, device
        )
        
        final_metrics = compute_metrics(val_labels, val_preds, val_probs)
        
        # Store fold results
        fold_result = {
            'fold': fold_idx,
            'best_epoch': best_epoch,
            'train_patients': len(fold_data['train_patients']),
            'val_patients': len(fold_data['val_patients']),
            'best_val_acc': val_acc,
            'best_val_f1': final_metrics['f1_weighted'],
            'metrics': final_metrics,
            'history': fold_history
        }
        results.append(fold_result)
        
        # Print fold summary
        print(f"\n{'='*60}")
        print(f"Fold {fold_idx} Final Results:")
        print(f"{'='*60}")
        print(f"Best Epoch: {best_epoch}")
        print(f"Accuracy: {val_acc:.4f}")
        print(f"F1-Score (weighted): {final_metrics['f1_weighted']:.4f}")
        if final_metrics['auc'] is not None:
            print(f"AUC: {final_metrics['auc']:.4f}")
        
        print(f"\nPer-Class Metrics:")
        for i, class_name in enumerate(final_metrics['class_names']):
            print(f"  {class_name}:")
            print(f"    Precision: {final_metrics['precision_per_class'][i]:.4f}")
            print(f"    Recall: {final_metrics['recall_per_class'][i]:.4f}")
            print(f"    F1-Score: {final_metrics['f1_per_class'][i]:.4f}")
            print(f"    Support: {final_metrics['support_per_class'][i]}")
        
        print(f"\nConfusion Matrix:")
        cm = np.array(final_metrics['confusion_matrix'])
        print(f"              Pred MF  Pred Non-MF")
        print(f"  True MF     {cm[0,0]:6d}  {cm[0,1]:11d}")
        print(f"  True Non-MF {cm[1,0]:6d}  {cm[1,1]:11d}")
        print(f"{'='*60}\n")
    
    return results

# Train models for both magnifications
print("\n" + "="*80)
print("TRAINING WITH 5-FOLD CROSS-VALIDATION")
print("="*80)

# Train x20 model
results_20 = train_kfold_cv(
    folds_20, 
    mag_name='x20',
    model_name=model_configs['x20']['model_name'],
    epochs=model_configs['x20']['epochs'],
    batch_size=model_configs['x20']['batch_size'],
    lr=model_configs['x20']['lr'],
    dropout=model_configs['x20']['dropout']
)

# Train x10 model
results_10 = train_kfold_cv(
    folds_10,
    mag_name='x10', 
    model_name=model_configs['x10']['model_name'],
    epochs=model_configs['x10']['epochs'],
    batch_size=model_configs['x10']['batch_size'],
    lr=model_configs['x10']['lr'],
    dropout=model_configs['x10']['dropout']
)

# Save results
with open(save_path / 'cv_results_x20.json', 'w') as f:
    json.dump(results_20, f, indent=2)
with open(save_path / 'cv_results_x10.json', 'w') as f:
    json.dump(results_10, f, indent=2)

print("\n✓ Results saved to:", save_path)


TRAINING WITH 5-FOLD CROSS-VALIDATION

Starting 5-Fold Cross-Validation for x20
Model: tf_efficientnet_b2 | Epochs: 5 | Batch Size: 8 | LR: 0.0003


Fold 1/5 - x20
Train patches: 23706, Val patches: 4968
Train patients: 28, Val patients: 8
  Class distribution: {0: 16038, 1: 7668}
  Class weights: ['1.4781', '3.0915']


C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_20648\2417676137.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/5 | train_loss 0.3909 train_acc 0.8610 | val_loss 0.4427 val_acc 0.8547 val_f1 0.8568 | time 753.3s
  → Saved best model (val_f1: 0.8568)


C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_20648\2417676137.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 2/5 | train_loss 0.1798 train_acc 0.9347 | val_loss 0.3518 val_acc 0.8780 val_f1 0.8775 | time 706.5s
  → Saved best model (val_f1: 0.8775)


...

Epoch 4/5 | train_loss 0.1321 train_acc 0.9501 | val_loss 1.0398 val_acc 0.6808 val_f1 0.6816 | time 289.1s


C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_20648\2417676137.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 5/5 | train_loss 0.1144 train_acc 0.9569 | val_loss 1.0010 val_acc 0.7515 val_f1 0.7520 | time 284.5s
  → Saved best model (val_f1: 0.7520)

Loading best model from epoch 5...


C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_20648\2417676137.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(save_path / f'model_{model_name}_{mag_n


Fold 5 Final Results:
Best Epoch: 5
Accuracy: 0.7515
F1-Score (weighted): 0.7520
AUC: 0.8226

Per-Class Metrics:
  MF:
    Precision: 0.7921
    Recall: 0.7460
    F1-Score: 0.7684
    Support: 1134
  Non-MF:
    Precision: 0.7073
    Recall: 0.7582
    F1-Score: 0.7319
    Support: 918

Confusion Matrix:
              Pred MF  Pred Non-MF
  True MF        846          288
  True Non-MF    222          696


✓ Results saved to: G:\My Drive\CLPD-MF-Dataset\Local Models\CV_Results


In [10]:
# Cell 8 - Comprehensive Results Analysis and Comparison
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

save_path = Path("G:/My Drive/CLPD-MF-Dataset/Local Models/CV_Results")

def load_cv_results(mag):
    """Load CV results from JSON"""
    results_file = save_path / f'cv_results_{mag}.json'
    with open(results_file, 'r') as f:
        return json.load(f)

def print_cv_summary(results, mag_name):
    """Print comprehensive CV summary statistics"""
    print(f"\n{'='*80}")
    print(f"{mag_name} - 5-FOLD CROSS-VALIDATION RESULTS SUMMARY")
    print(f"{'='*80}")
    
    # Extract metrics across folds
    accuracies = [r['best_val_acc'] for r in results]
    f1_scores = [r['best_val_f1'] for r in results]
    
    # Per-class metrics
    mf_precisions = [r['metrics']['precision_per_class'][0] for r in results]
    mf_recalls = [r['metrics']['recall_per_class'][0] for r in results]
    mf_f1s = [r['metrics']['f1_per_class'][0] for r in results]
    
    nonmf_precisions = [r['metrics']['precision_per_class'][1] for r in results]
    nonmf_recalls = [r['metrics']['recall_per_class'][1] for r in results]
    nonmf_f1s = [r['metrics']['f1_per_class'][1] for r in results]
    
    # AUC (if available)
    aucs = [r['metrics']['auc'] for r in results if r['metrics']['auc'] is not None]
    
    # Calculate confidence intervals (95%)
    def ci_95(data):
        mean = np.mean(data)
        std = np.std(data, ddof=1)
        n = len(data)
        se = std / np.sqrt(n)
        ci = 1.96 * se  # 95% CI
        return mean, std, ci
    
    # Overall metrics
    print(f"\n📊 OVERALL PERFORMANCE:")
    print(f"{'Metric':<25} {'Mean':<12} {'Std':<12} {'95% CI':<20} {'Range'}")
    print("-" * 80)
    
    acc_mean, acc_std, acc_ci = ci_95(accuracies)
    print(f"{'Accuracy':<25} {acc_mean:.4f}      ±{acc_std:.4f}     "
          f"[{acc_mean-acc_ci:.4f}, {acc_mean+acc_ci:.4f}]   "
          f"[{min(accuracies):.4f}, {max(accuracies):.4f}]")
    
    f1_mean, f1_std, f1_ci = ci_95(f1_scores)
    print(f"{'F1-Score (weighted)':<25} {f1_mean:.4f}      ±{f1_std:.4f}     "
          f"[{f1_mean-f1_ci:.4f}, {f1_mean+f1_ci:.4f}]   "
          f"[{min(f1_scores):.4f}, {max(f1_scores):.4f}]")
    
    if aucs:
        auc_mean, auc_std, auc_ci = ci_95(aucs)
        print(f"{'AUC':<25} {auc_mean:.4f}      ±{auc_std:.4f}     "
              f"[{auc_mean-auc_ci:.4f}, {auc_mean+auc_ci:.4f}]   "
              f"[{min(aucs):.4f}, {max(aucs):.4f}]")
    
    # MF class (minority class - most important!)
    print(f"\n🎯 MF CLASS PERFORMANCE (Critical for Medical Diagnosis):")
    print(f"{'Metric':<25} {'Mean':<12} {'Std':<12} {'95% CI':<20} {'Range'}")
    print("-" * 80)
    
    mf_prec_mean, mf_prec_std, mf_prec_ci = ci_95(mf_precisions)
    print(f"{'Precision':<25} {mf_prec_mean:.4f}      ±{mf_prec_std:.4f}     "
          f"[{mf_prec_mean-mf_prec_ci:.4f}, {mf_prec_mean+mf_prec_ci:.4f}]   "
          f"[{min(mf_precisions):.4f}, {max(mf_precisions):.4f}]")
    
    mf_rec_mean, mf_rec_std, mf_rec_ci = ci_95(mf_recalls)
    print(f"{'Recall (Sensitivity)':<25} {mf_rec_mean:.4f}      ±{mf_rec_std:.4f}     "
          f"[{mf_rec_mean-mf_rec_ci:.4f}, {mf_rec_mean+mf_rec_ci:.4f}]   "
          f"[{min(mf_recalls):.4f}, {max(mf_recalls):.4f}]")
    
    mf_f1_mean, mf_f1_std, mf_f1_ci = ci_95(mf_f1s)
    print(f"{'F1-Score':<25} {mf_f1_mean:.4f}      ±{mf_f1_std:.4f}     "
          f"[{mf_f1_mean-mf_f1_ci:.4f}, {mf_f1_mean+mf_f1_ci:.4f}]   "
          f"[{min(mf_f1s):.4f}, {max(mf_f1s):.4f}]")
    
    # Non-MF class
    print(f"\n✅ NON-MF CLASS PERFORMANCE:")
    print(f"{'Metric':<25} {'Mean':<12} {'Std':<12} {'95% CI':<20} {'Range'}")
    print("-" * 80)
    
    nonmf_prec_mean, nonmf_prec_std, nonmf_prec_ci = ci_95(nonmf_precisions)
    print(f"{'Precision':<25} {nonmf_prec_mean:.4f}      ±{nonmf_prec_std:.4f}     "
          f"[{nonmf_prec_mean-nonmf_prec_ci:.4f}, {nonmf_prec_mean+nonmf_prec_ci:.4f}]   "
          f"[{min(nonmf_precisions):.4f}, {max(nonmf_precisions):.4f}]")
    
    nonmf_rec_mean, nonmf_rec_std, nonmf_rec_ci = ci_95(nonmf_recalls)
    print(f"{'Recall (Specificity)':<25} {nonmf_rec_mean:.4f}      ±{nonmf_rec_std:.4f}     "
          f"[{nonmf_rec_mean-nonmf_rec_ci:.4f}, {nonmf_rec_mean+nonmf_rec_ci:.4f}]   "
          f"[{min(nonmf_recalls):.4f}, {max(nonmf_recalls):.4f}]")
    
    nonmf_f1_mean, nonmf_f1_std, nonmf_f1_ci = ci_95(nonmf_f1s)
    print(f"{'F1-Score':<25} {nonmf_f1_mean:.4f}      ±{nonmf_f1_std:.4f}     "
          f"[{nonmf_f1_mean-nonmf_f1_ci:.4f}, {nonmf_f1_mean+nonmf_f1_ci:.4f}]   "
          f"[{min(nonmf_f1s):.4f}, {max(nonmf_f1s):.4f}]")
    
    # Per-fold details
    print(f"\n📋 PER-FOLD BREAKDOWN:")
    print(f"{'Fold':<8} {'Accuracy':<12} {'F1-Score':<12} {'MF Recall':<12} {'Non-MF Recall':<15} {'Patients (Val)'}")
    print("-" * 80)
    
    for r in results:
        fold = r['fold']
        acc = r['best_val_acc']
        f1 = r['best_val_f1']
        mf_rec = r['metrics']['recall_per_class'][0]
        nonmf_rec = r['metrics']['recall_per_class'][1]
        n_patients = r['val_patients']
        
        print(f"{fold:<8} {acc:.4f}       {f1:.4f}       {mf_rec:.4f}       "
              f"{nonmf_rec:.4f}          {n_patients}")
    
    # Aggregated confusion matrix
    print(f"\n📊 AGGREGATED CONFUSION MATRIX (Sum Across All Folds):")
    total_cm = np.zeros((2, 2), dtype=int)
    for r in results:
        total_cm += np.array(r['metrics']['confusion_matrix'])
    
    print(f"                Predicted MF  Predicted Non-MF")
    print(f"  Actual MF          {total_cm[0,0]:6d}         {total_cm[0,1]:12d}")
    print(f"  Actual Non-MF      {total_cm[1,0]:6d}         {total_cm[1,1]:12d}")
    
    # False negative and false positive rates
    total_mf = total_cm[0,0] + total_cm[0,1]
    total_nonmf = total_cm[1,0] + total_cm[1,1]
    fnr = total_cm[0,1] / total_mf if total_mf > 0 else 0
    fpr = total_cm[1,0] / total_nonmf if total_nonmf > 0 else 0
    
    print(f"\n  False Negative Rate (missed MF): {fnr:.2%} ({total_cm[0,1]}/{total_mf})")
    print(f"  False Positive Rate (false alarms): {fpr:.2%} ({total_cm[1,0]}/{total_nonmf})")
    
    print(f"{'='*80}\n")
    
    return {
        'accuracy': (acc_mean, acc_std, acc_ci),
        'f1_score': (f1_mean, f1_std, f1_ci),
        'mf_recall': (mf_rec_mean, mf_rec_std, mf_rec_ci),
        'mf_precision': (mf_prec_mean, mf_prec_std, mf_prec_ci)
    }

def compare_with_previous_results(cv_results, mag_name, previous_acc):
    """Compare CV results with your previous 4-patient test set"""
    print(f"\n{'='*80}")
    print(f"COMPARISON: CV vs Previous 4-Patient Test Set ({mag_name})")
    print(f"{'='*80}\n")
    
    cv_acc = np.mean([r['best_val_acc'] for r in cv_results])
    cv_std = np.std([r['best_val_acc'] for r in cv_results])
    
    print(f"Previous Test Set (n=4 patients):")
    print(f"  Accuracy: {previous_acc:.2%}")
    print(f"  ⚠️  Confidence Interval: N/A (sample too small)")
    print(f"  ⚠️  Statistical Power: Insufficient")
    
    print(f"\n5-Fold Cross-Validation:")
    print(f"  Accuracy: {cv_acc:.2%} ± {cv_std:.2%}")
    print(f"  ✓ Tests on ~{cv_results[0]['val_patients']*5} total patient validations")
    print(f"  ✓ 95% CI: [{cv_acc - 1.96*cv_std/np.sqrt(5):.2%}, {cv_acc + 1.96*cv_std/np.sqrt(5):.2%}]")
    print(f"  ✓ Robust estimate across different data splits")
    
    print(f"\n💡 Interpretation:")
    if abs(cv_acc - previous_acc) > 0.10:
        print(f"  → Large difference ({abs(cv_acc - previous_acc):.1%}) suggests previous")
        print(f"     result was likely due to small sample size")
    else:
        print(f"  → Difference is reasonable ({abs(cv_acc - previous_acc):.1%})")
    
    print(f"\n  The CV result is MORE RELIABLE because:")
    print(f"  1. Larger effective test set (~{cv_results[0]['val_patients']} patients per fold)")
    print(f"  2. Multiple independent evaluations (5 folds)")
    print(f"  3. Confidence intervals quantify uncertainty")
    print(f"  4. Less susceptible to lucky/unlucky splits")
    
    print(f"{'='*80}\n")

# Load and analyze results
results_20 = load_cv_results('x20')
results_10 = load_cv_results('x10')

# Print summaries
summary_20 = print_cv_summary(results_20, 'x20')
summary_10 = print_cv_summary(results_10, 'x10')

# Compare with your previous results
# EfficientNetB2 previous results: x20=100%, x10=100% (but only 4 patients!)
print("\n" + "="*80)
print("COMPARISON WITH PREVIOUS RESULTS")
print("="*80)

compare_with_previous_results(results_20, 'x20', previous_acc=1.00)
compare_with_previous_results(results_10, 'x10', previous_acc=1.00)

# Final recommendation
print(f"\n{'='*80}")
print("🎯 FINAL ASSESSMENT & RECOMMENDATIONS")
print(f"{'='*80}\n")

x20_acc_mean = np.mean([r['best_val_acc'] for r in results_20])
x10_acc_mean = np.mean([r['best_val_acc'] for r in results_10])
x20_mf_recall = np.mean([r['metrics']['recall_per_class'][0] for r in results_20])
x10_mf_recall = np.mean([r['metrics']['recall_per_class'][0] for r in results_10])

print(f"Model Performance Summary:")
print(f"  x20: {x20_acc_mean:.1%} accuracy, {x20_mf_recall:.1%} MF recall")
print(f"  x10: {x10_acc_mean:.1%} accuracy, {x10_mf_recall:.1%} MF recall")

if x20_acc_mean >= 0.85 and x20_mf_recall >= 0.80:
    print(f"\n✅ Performance is GOOD for medical imaging with limited data")
    print(f"   → Consider this model ready for further validation")
    print(f"   → Could proceed to external test set if available")
elif x20_acc_mean >= 0.75:
    print(f"\n⚠️  Performance is MODERATE")
    print(f"   → Consider: more data augmentation, ensemble methods")
    print(f"   → May need more training data")
else:
    print(f"\n❌ Performance is BELOW TARGET")
    print(f"   → Need: data quality review, more samples, or different approach")

print(f"\nShould you try Swin Transformer?")
if x20_acc_mean < 0.80:
    print(f"  → YES, might help improve performance")
    print(f"  → But also try: stronger regularization, better augmentation first")
else:
    print(f"  → MAYBE NOT NECESSARY - EfficientNet is working well")
    print(f"  → Focus on: data collection, ensemble, or deployment instead")
    print(f"  → Swin needs more data (1000+ images) to shine")

print(f"\n{'='*80}\n")


x20 - 5-FOLD CROSS-VALIDATION RESULTS SUMMARY

📊 OVERALL PERFORMANCE:
Metric                    Mean         Std          95% CI               Range
--------------------------------------------------------------------------------
Accuracy                  0.8726      ±0.0457     [0.8326, 0.9126]   [0.8230, 0.9275]
F1-Score (weighted)       0.8743      ±0.0451     [0.8348, 0.9138]   [0.8205, 0.9285]
AUC                       0.9262      ±0.0555     [0.8776, 0.9748]   [0.8330, 0.9803]

🎯 MF CLASS PERFORMANCE (Critical for Medical Diagnosis):
Metric                    Mean         Std          95% CI               Range
--------------------------------------------------------------------------------
Precision                 0.9170      ±0.0591     [0.8652, 0.9687]   [0.8526, 0.9817]
Recall (Sensitivity)      0.8909      ±0.0436     [0.8526, 0.9291]   [0.8282, 0.9486]
F1-Score                  0.9023      ±0.0334     [0.8731, 0.9316]   [0.8713, 0.9421]

✅ NON-MF CLASS PERFORMANCE:
Metric

In [ ]:
# Patient-Level Testing Cell for x10 Model
import torch
import numpy as np
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# 1. Load the Previously Trained x10 Model
# ============================================================================

def load_pretrained_model(model_path, model_name='resnet50', num_classes=2):
    """Load a previously trained model from disk"""
    print(f"Loading model from: {model_path}")
    
    model = create_model(
        model_name=model_name,
        pretrained=False,  # Don't load ImageNet weights
        num_classes=num_classes,
        dropout=0.3
    )
    
    # Load saved weights
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    print(f"✓ Model loaded successfully")
    return model

# Load your previously trained x10 model
model_path = Path(r"C:/Users/Mohamed Hazem/Graduation Project/Dr. Rushdy/CLPD Dr. Kariman/MF Clasasifier/Trained Models/model_resnet50_x10.pth")

if not model_path.exists():
    raise FileNotFoundError(f"Model not found at: {model_path}")

x10_model = load_pretrained_model(
    model_path, 
    model_name='resnet50',  # Your previous model was ResNet50
    num_classes=2
)

print("="*80)

# ============================================================================
# 2. Patient-Level Prediction Function
# ============================================================================

def predict_patient(patient_rows, model, device, transforms=val_tf, aggregation='mean'):
    """
    Predict diagnosis for one patient by aggregating predictions across all their images.
    
    Args:
        patient_rows: List of image rows for this patient
        model: Trained model
        device: torch device
        transforms: Image transforms
        aggregation: 'mean', 'majority', or 'max_confidence'
    
    Returns:
        dict with prediction, confidence, and per-image details
    """
    model.eval()
    
    all_probs = []
    all_preds = []
    image_details = []
    
    with torch.no_grad():
        for row in patient_rows:
            # Extract patches from this image
            patches = extract_and_cache_patches(
                row['path'], 
                patch_size=512, 
                stride=256, 
                max_patches_per_image=100
            )
            
            if len(patches) == 0:
                continue
            
            # Predict on all patches from this image
            patch_probs = []
            for patch_path in patches:
                img = Image.open(patch_path).convert('RGB')
                img_tensor = transforms(img).unsqueeze(0).to(device)
                
                outputs = model(img_tensor)
                probs = F.softmax(outputs, dim=1).cpu().numpy()[0]
                patch_probs.append(probs)
            
            # Average predictions across patches for this image
            image_prob = np.mean(patch_probs, axis=0)
            image_pred = image_prob.argmax()
            
            all_probs.append(image_prob)
            all_preds.append(image_pred)
            
            image_details.append({
                'image_name': row['path'].name,
                'num_patches': len(patches),
                'prob_MF': image_prob[0],
                'prob_NonMF': image_prob[1],
                'prediction': 'MF' if image_pred == 0 else 'Non-MF'
            })
    
    if len(all_probs) == 0:
        return None
    
    # Aggregate across all images for patient-level prediction
    if aggregation == 'mean':
        # Average probabilities across all images
        patient_prob = np.mean(all_probs, axis=0)
        patient_pred = patient_prob.argmax()
        confidence = patient_prob[patient_pred]
        
    elif aggregation == 'majority':
        # Majority vote of image predictions
        vote_counts = Counter(all_preds)
        patient_pred = vote_counts.most_common(1)[0][0]
        confidence = vote_counts[patient_pred] / len(all_preds)
        patient_prob = np.mean(all_probs, axis=0)  # Still compute for reference
        
    elif aggregation == 'max_confidence':
        # Take prediction from most confident image
        confidences = [max(prob) for prob in all_probs]
        max_idx = np.argmax(confidences)
        patient_prob = all_probs[max_idx]
        patient_pred = patient_prob.argmax()
        confidence = patient_prob[patient_pred]
    
    return {
        'prediction': patient_pred,
        'predicted_class': 'MF' if patient_pred == 0 else 'Non-MF',
        'confidence': float(confidence),
        'prob_MF': float(patient_prob[0]),
        'prob_NonMF': float(patient_prob[1]),
        'num_images': len(patient_rows),
        'num_predictions': len(all_preds),
        'image_details': image_details,
        'aggregation_method': aggregation
    }

# ============================================================================
# 3. Test on ALL Patients at x10
# ============================================================================

def test_all_patients_x10(all_images, model, mag='x10', aggregation='mean'):
    """
    Test model on all patients with x10 images.
    Returns patient-level predictions and metrics.
    """
    print(f"\n{'='*80}")
    print(f"PATIENT-LEVEL TESTING FOR x10 MODEL")
    print(f"Aggregation method: {aggregation}")
    print(f"{'='*80}\n")
    
    # Group images by patient
    x10_images = [r for r in all_images if r['mag'] == mag]
    
    # Organize by patient
    patient_groups = defaultdict(list)
    for row in x10_images:
        patient_groups[row['patient']].append(row)
    
    print(f"Total x10 patients: {len(patient_groups)}")
    
    # Get ground truth labels for each patient
    patient_true_labels = {}
    for patient, rows in patient_groups.items():
        # All images from same patient should have same label
        labels = set(r['label'] for r in rows)
        if len(labels) > 1:
            print(f"⚠️  Warning: Patient {patient} has mixed labels: {labels}")
        patient_true_labels[patient] = list(labels)[0]
    
    # Count patients per class
    mf_patients = sum(1 for label in patient_true_labels.values() if label == 'MF')
    nonmf_patients = len(patient_true_labels) - mf_patients
    print(f"  MF patients: {mf_patients}")
    print(f"  Non-MF patients: {nonmf_patients}")
    print()
    
    # Predict for each patient
    results = []
    y_true = []
    y_pred = []
    y_probs = []
    
    label_map = {'MF': 0, 'Non-MF': 1}
    
    for patient, rows in patient_groups.items():
        true_label = patient_true_labels[patient]
        true_label_idx = label_map[true_label]
        
        # Get patient-level prediction
        pred_result = predict_patient(rows, model, device, aggregation=aggregation)
        
        if pred_result is None:
            print(f"⚠️  Skipping {patient} - no valid patches")
            continue
        
        pred_label_idx = pred_result['prediction']
        
        # Store for metrics
        y_true.append(true_label_idx)
        y_pred.append(pred_label_idx)
        y_probs.append([pred_result['prob_MF'], pred_result['prob_NonMF']])
        
        # Store detailed results
        results.append({
            'patient': patient,
            'true_label': true_label,
            'predicted_label': pred_result['predicted_class'],
            'correct': true_label == pred_result['predicted_class'],
            'confidence': pred_result['confidence'],
            'prob_MF': pred_result['prob_MF'],
            'prob_NonMF': pred_result['prob_NonMF'],
            'num_images': pred_result['num_images'],
            'image_details': pred_result['image_details']
        })
    
    return results, y_true, y_pred, y_probs

# ============================================================================
# 4. Run Tests with Different Aggregation Methods
# ============================================================================

print("\n" + "="*80)
print("TESTING x10 MODEL WITH DIFFERENT AGGREGATION STRATEGIES")
print("="*80)

aggregation_methods = ['mean', 'majority', 'max_confidence']
all_method_results = {}

for agg_method in aggregation_methods:
    results, y_true, y_pred, y_probs = test_all_patients_x10(
        all_images, 
        x10_model, 
        mag='x10', 
        aggregation=agg_method
    )
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=[0, 1], zero_division=0
    )
    
    # Weighted metrics
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    # AUC
    try:
        y_probs_array = np.array(y_probs)
        auc = roc_auc_score(y_true, y_probs_array[:, 1])
    except:
        auc = None
    
    # Store results
    all_method_results[agg_method] = {
        'accuracy': accuracy,
        'precision_per_class': precision,
        'recall_per_class': recall,
        'f1_per_class': f1,
        'support': support,
        'precision_weighted': precision_w,
        'recall_weighted': recall_w,
        'f1_weighted': f1_w,
        'confusion_matrix': cm,
        'auc': auc,
        'results': results
    }
    
    # Print results
    print(f"\n{'='*80}")
    print(f"RESULTS - Aggregation Method: {agg_method.upper()}")
    print(f"{'='*80}")
    
    print(f"\n📊 Overall Metrics:")
    print(f"  Accuracy: {accuracy:.4f} ({int(accuracy*len(y_true))}/{len(y_true)} patients correct)")
    print(f"  F1-Score (weighted): {f1_w:.4f}")
    if auc is not None:
        print(f"  AUC: {auc:.4f}")
    
    print(f"\n🎯 Per-Class Metrics:")
    class_names = ['MF', 'Non-MF']
    for i, cls in enumerate(class_names):
        print(f"  {cls}:")
        print(f"    Precision: {precision[i]:.4f}")
        print(f"    Recall: {recall[i]:.4f}")
        print(f"    F1-Score: {f1[i]:.4f}")
        print(f"    Support: {support[i]} patients")
    
    print(f"\n📋 Confusion Matrix:")
    print(f"                Predicted MF  Predicted Non-MF")
    print(f"  Actual MF          {cm[0,0]:6d}         {cm[0,1]:12d}")
    print(f"  Actual Non-MF      {cm[1,0]:6d}         {cm[1,1]:12d}")
    
    # Calculate error rates
    if support[0] > 0:
        fnr = cm[0,1] / support[0]  # False negative rate
        print(f"\n  ⚠️  False Negative Rate: {fnr:.2%} ({cm[0,1]}/{support[0]} MF patients missed)")
    
    if support[1] > 0:
        fpr = cm[1,0] / support[1]  # False positive rate
        print(f"  ⚠️  False Positive Rate: {fpr:.2%} ({cm[1,0]}/{support[1]} Non-MF patients misclassified)")
    
    # Show misclassified patients
    misclassified = [r for r in results if not r['correct']]
    if misclassified:
        print(f"\n❌ Misclassified Patients ({len(misclassified)}):")
        for r in misclassified:
            print(f"  • {r['patient']}")
            print(f"      True: {r['true_label']} | Predicted: {r['predicted_label']} | "
                  f"Confidence: {r['confidence']:.2%} | Images: {r['num_images']}")

# ============================================================================
# 5. Comparison Summary
# ============================================================================

print(f"\n\n{'='*80}")
print("COMPARISON OF AGGREGATION METHODS")
print(f"{'='*80}\n")

print(f"{'Method':<20} {'Accuracy':<12} {'F1-Score':<12} {'MF Recall':<12} {'AUC':<12}")
print("-"*80)

for method in aggregation_methods:
    res = all_method_results[method]
    acc = res['accuracy']
    f1 = res['f1_weighted']
    mf_recall = res['recall_per_class'][0]
    auc = res['auc'] if res['auc'] is not None else 0.0
    
    print(f"{method:<20} {acc:.4f}       {f1:.4f}       {mf_recall:.4f}       {auc:.4f}")

# ============================================================================
# 6. Final Recommendation
# ============================================================================

print(f"\n\n{'='*80}")
print("🎯 FINAL RECOMMENDATION FOR x10 MODEL")
print(f"{'='*80}\n")

# Find best aggregation method
best_method = max(aggregation_methods, 
                  key=lambda m: all_method_results[m]['f1_weighted'])
best_results = all_method_results[best_method]

print(f"Best Aggregation Method: {best_method.upper()}")
print(f"  Patient-Level Accuracy: {best_results['accuracy']:.2%}")
print(f"  MF Recall (Sensitivity): {best_results['recall_per_class'][0]:.2%}")
print(f"  Non-MF Recall (Specificity): {best_results['recall_per_class'][1]:.2%}")

# Decision criteria
mf_recall = best_results['recall_per_class'][0]
accuracy = best_results['accuracy']

print(f"\n💡 Clinical Decision:")

if mf_recall >= 0.85 and accuracy >= 0.80:
    print(f"  ✅ INCLUDE x10 in your ensemble")
    print(f"     → Performance is good enough for clinical support")
    print(f"     → Can complement x20 predictions")
    print(f"     → Use {best_method} aggregation for patient-level predictions")
elif mf_recall >= 0.75 and accuracy >= 0.70:
    print(f"  ⚠️  CONDITIONALLY INCLUDE x10")
    print(f"     → Performance is acceptable but not great")
    print(f"     → Use only as SECONDARY input to x20")
    print(f"     → Flag low-confidence predictions for review")
    print(f"     → Consider: x20 vote counts more (e.g., 70% x20, 30% x10)")
else:
    print(f"  ❌ DO NOT INCLUDE x10 in clinical deployment")
    print(f"     → Performance too unreliable for patient care")
    print(f"     → Missing too many MF cases ({100-mf_recall*100:.1f}%)")
    print(f"     → Use only x20 model for now")
    print(f"     → Collect more x10 data before reconsidering")

# Comparison with CV results
print(f"\n📊 Comparison with 5-Fold CV Patch-Level Results:")
print(f"  CV Mean Accuracy: 73.8% (patch-level)")
print(f"  Patient-Level Accuracy: {accuracy:.1%} (patient-level)")
print(f"  ")
if accuracy > 0.738:
    print(f"  ✓ Patient-level aggregation improves performance!")
    print(f"    → Multiple images per patient provides redundancy")
else:
    print(f"  → Similar or worse than patch-level CV")
    print(f"    → Model struggles even with multiple images")

print(f"\n{'='*80}\n")

# Save detailed results for further analysis
import json
output_path = Path("G:/My Drive/CLPD-MF-Dataset/Local Models/Hazem/CV_Results/x10_patient_level_test.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_data = {
    'model_path': str(model_path),
    'aggregation_methods': {
        method: {
            'accuracy': float(res['accuracy']),
            'f1_weighted': float(res['f1_weighted']),
            'mf_recall': float(res['recall_per_class'][0]),
            'nonmf_recall': float(res['recall_per_class'][1]),
            'confusion_matrix': res['confusion_matrix'].tolist(),
            'auc': float(res['auc']) if res['auc'] is not None else None
        }
        for method, res in all_method_results.items()
    },
    'best_method': best_method,
    'recommendation': 'include' if mf_recall >= 0.85 and accuracy >= 0.80 else 
                     'conditional' if mf_recall >= 0.75 and accuracy >= 0.70 else 'exclude'
}

with open(output_path, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"✓ Detailed results saved to: {output_path}")

Loading model from: C:\Users\Mohamed Hazem\Graduation Project\Dr. Rushdy\CLPD Dr. Kariman\MF Clasasifier\Trained Models\model_resnet50_x10.pth


C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_12252\3976851989.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=de

✓ Model loaded successfully

TESTING x10 MODEL WITH DIFFERENT AGGREGATION STRATEGIES

PATIENT-LEVEL TESTING FOR x10 MODEL
Aggregation method: mean

Total x10 patients: 41
  MF patients: 10
  Non-MF patients: 31


RESULTS - Aggregation Method: MEAN

📊 Overall Metrics:
  Accuracy: 0.9756 (40/41 patients correct)
  F1-Score (weighted): 0.9752
  AUC: 0.9968

🎯 Per-Class Metrics:
  MF:
    Precision: 1.0000
    Recall: 0.9000
    F1-Score: 0.9474
    Support: 10 patients
  Non-MF:
    Precision: 0.9688
    Recall: 1.0000
    F1-Score: 0.9841
    Support: 31 patients

📋 Confusion Matrix:
                Predicted MF  Predicted Non-MF
  Actual MF               9                    1
  Actual Non-MF           0                   31

  ⚠️  False Negative Rate: 10.00% (1/10 MF patients missed)
  ⚠️  False Positive Rate: 0.00% (0/31 Non-MF patients misclassified)

❌ Misclassified Patients (1):
  • maickel nasery 19-20-12-21
      True: MF | Predicted: Non-MF | Confidence: 62.24% | Images: 16

PAT

In [ ]:
# Patient-Level Testing Cell for x20 Model ResNet50
import torch
import numpy as np
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)
import torch.nn.functional as F

device = torch.device('cuda')

# ============================================================================
# 1. Load the Previously Trained x20 Model
# ============================================================================

def load_pretrained_model(model_path, model_name='resnet50', num_classes=2):
    """Load a previously trained model from disk"""
    print(f"Loading model from: {model_path}")
    
    model = create_model(
        model_name=model_name,
        pretrained=False,  # Don't load ImageNet weights
        num_classes=num_classes,
        dropout=0.3
    )
    
    # Load saved weights
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    print(f"✓ Model loaded successfully")
    return model

# Load your previously trained x20 model
model_path = Path(r"C:/Users/Mohamed Hazem/Graduation Project/Dr. Rushdy/CLPD Dr. Kariman/MF Clasasifier/Trained Models/model_resnet50_x20.pth")

if not model_path.exists():
    raise FileNotFoundError(f"Model not found at: {model_path}")

x20_model = load_pretrained_model(
    model_path, 
    model_name='resnet50',  # Your previous model was ResNet50
    num_classes=2
)

print("="*80)

# ============================================================================
# 2. Patient-Level Prediction Function
# ============================================================================

def predict_patient(patient_rows, model, device, transforms=val_tf, aggregation='mean'):
    """
    Predict diagnosis for one patient by aggregating predictions across all their images.
    
    Args:
        patient_rows: List of image rows for this patient
        model: Trained model
        device: torch device
        transforms: Image transforms
        aggregation: 'mean', 'majority', or 'max_confidence'
    
    Returns:
        dict with prediction, confidence, and per-image details
    """
    model.eval()
    
    all_probs = []
    all_preds = []
    image_details = []
    
    with torch.no_grad():
        for row in patient_rows:
            # Extract patches from this image
            patches = extract_and_cache_patches(
                row['path'], 
                patch_size=512, 
                stride=256, 
                max_patches_per_image=100
            )
            
            if len(patches) == 0:
                continue
            
            # Predict on all patches from this image
            patch_probs = []
            for patch_path in patches:
                img = Image.open(patch_path).convert('RGB')
                img_tensor = transforms(img).unsqueeze(0).to(device)
                
                outputs = model(img_tensor)
                probs = F.softmax(outputs, dim=1).cpu().numpy()[0]
                patch_probs.append(probs)
            
            # Average predictions across patches for this image
            image_prob = np.mean(patch_probs, axis=0)
            image_pred = image_prob.argmax()
            
            all_probs.append(image_prob)
            all_preds.append(image_pred)
            
            image_details.append({
                'image_name': row['path'].name,
                'num_patches': len(patches),
                'prob_MF': image_prob[0],
                'prob_NonMF': image_prob[1],
                'prediction': 'MF' if image_pred == 0 else 'Non-MF'
            })
    
    if len(all_probs) == 0:
        return None
    
    # Aggregate across all images for patient-level prediction
    if aggregation == 'mean':
        # Average probabilities across all images
        patient_prob = np.mean(all_probs, axis=0)
        patient_pred = patient_prob.argmax()
        confidence = patient_prob[patient_pred]
        
    elif aggregation == 'majority':
        # Majority vote of image predictions
        vote_counts = Counter(all_preds)
        patient_pred = vote_counts.most_common(1)[0][0]
        confidence = vote_counts[patient_pred] / len(all_preds)
        patient_prob = np.mean(all_probs, axis=0)  # Still compute for reference
        
    elif aggregation == 'max_confidence':
        # Take prediction from most confident image
        confidences = [max(prob) for prob in all_probs]
        max_idx = np.argmax(confidences)
        patient_prob = all_probs[max_idx]
        patient_pred = patient_prob.argmax()
        confidence = patient_prob[patient_pred]
    
    return {
        'prediction': patient_pred,
        'predicted_class': 'MF' if patient_pred == 0 else 'Non-MF',
        'confidence': float(confidence),
        'prob_MF': float(patient_prob[0]),
        'prob_NonMF': float(patient_prob[1]),
        'num_images': len(patient_rows),
        'num_predictions': len(all_preds),
        'image_details': image_details,
        'aggregation_method': aggregation
    }

# ============================================================================
# 3. Test on ALL Patients at x20
# ============================================================================

def test_all_patients_x20(all_images, model, mag='x20', aggregation='mean'):
    """
    Test model on all patients with x20 images.
    Returns patient-level predictions and metrics.
    """
    print(f"\n{'='*80}")
    print(f"PATIENT-LEVEL TESTING FOR x20 MODEL")
    print(f"Aggregation method: {aggregation}")
    print(f"{'='*80}\n")
    
    # Group images by patient
    x20_images = [r for r in all_images if r['mag'] == mag]
    
    # Organize by patient
    patient_groups = defaultdict(list)
    for row in x20_images:
        patient_groups[row['patient']].append(row)
    
    print(f"Total x20 patients: {len(patient_groups)}")
    
    # Get ground truth labels for each patient
    patient_true_labels = {}
    for patient, rows in patient_groups.items():
        # All images from same patient should have same label
        labels = set(r['label'] for r in rows)
        if len(labels) > 1:
            print(f"⚠️  Warning: Patient {patient} has mixed labels: {labels}")
        patient_true_labels[patient] = list(labels)[0]
    
    # Count patients per class
    mf_patients = sum(1 for label in patient_true_labels.values() if label == 'MF')
    nonmf_patients = len(patient_true_labels) - mf_patients
    print(f"  MF patients: {mf_patients}")
    print(f"  Non-MF patients: {nonmf_patients}")
    print()
    
    # Predict for each patient
    results = []
    y_true = []
    y_pred = []
    y_probs = []
    
    label_map = {'MF': 0, 'Non-MF': 1}
    
    for patient, rows in patient_groups.items():
        true_label = patient_true_labels[patient]
        true_label_idx = label_map[true_label]
        
        # Get patient-level prediction
        pred_result = predict_patient(rows, model, device, aggregation=aggregation)
        
        if pred_result is None:
            print(f"⚠️  Skipping {patient} - no valid patches")
            continue
        
        pred_label_idx = pred_result['prediction']
        
        # Store for metrics
        y_true.append(true_label_idx)
        y_pred.append(pred_label_idx)
        y_probs.append([pred_result['prob_MF'], pred_result['prob_NonMF']])
        
        # Store detailed results
        results.append({
            'patient': patient,
            'true_label': true_label,
            'predicted_label': pred_result['predicted_class'],
            'correct': true_label == pred_result['predicted_class'],
            'confidence': pred_result['confidence'],
            'prob_MF': pred_result['prob_MF'],
            'prob_NonMF': pred_result['prob_NonMF'],
            'num_images': pred_result['num_images'],
            'image_details': pred_result['image_details']
        })
    
    return results, y_true, y_pred, y_probs

# ============================================================================
# 4. Run Tests with Different Aggregation Methods
# ============================================================================

print("\n" + "="*80)
print("TESTING x20 MODEL WITH DIFFERENT AGGREGATION STRATEGIES")
print("="*80)

aggregation_methods = ['mean', 'majority', 'max_confidence']
all_method_results = {}

for agg_method in aggregation_methods:
    results, y_true, y_pred, y_probs = test_all_patients_x10(
        all_images, 
        x20_model, 
        mag='x20', 
        aggregation=agg_method
    )
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=[0, 1], zero_division=0
    )
    
    # Weighted metrics
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    # AUC
    try:
        y_probs_array = np.array(y_probs)
        auc = roc_auc_score(y_true, y_probs_array[:, 1])
    except:
        auc = None
    
    # Store results
    all_method_results[agg_method] = {
        'accuracy': accuracy,
        'precision_per_class': precision,
        'recall_per_class': recall,
        'f1_per_class': f1,
        'support': support,
        'precision_weighted': precision_w,
        'recall_weighted': recall_w,
        'f1_weighted': f1_w,
        'confusion_matrix': cm,
        'auc': auc,
        'results': results
    }
    
    # Print results
    print(f"\n{'='*80}")
    print(f"RESULTS - Aggregation Method: {agg_method.upper()}")
    print(f"{'='*80}")
    
    print(f"\n📊 Overall Metrics:")
    print(f"  Accuracy: {accuracy:.4f} ({int(accuracy*len(y_true))}/{len(y_true)} patients correct)")
    print(f"  F1-Score (weighted): {f1_w:.4f}")
    if auc is not None:
        print(f"  AUC: {auc:.4f}")
    
    print(f"\n🎯 Per-Class Metrics:")
    class_names = ['MF', 'Non-MF']
    for i, cls in enumerate(class_names):
        print(f"  {cls}:")
        print(f"    Precision: {precision[i]:.4f}")
        print(f"    Recall: {recall[i]:.4f}")
        print(f"    F1-Score: {f1[i]:.4f}")
        print(f"    Support: {support[i]} patients")
    
    print(f"\n📋 Confusion Matrix:")
    print(f"                Predicted MF  Predicted Non-MF")
    print(f"  Actual MF          {cm[0,0]:6d}         {cm[0,1]:12d}")
    print(f"  Actual Non-MF      {cm[1,0]:6d}         {cm[1,1]:12d}")
    
    # Calculate error rates
    if support[0] > 0:
        fnr = cm[0,1] / support[0]  # False negative rate
        print(f"\n  ⚠️  False Negative Rate: {fnr:.2%} ({cm[0,1]}/{support[0]} MF patients missed)")
    
    if support[1] > 0:
        fpr = cm[1,0] / support[1]  # False positive rate
        print(f"  ⚠️  False Positive Rate: {fpr:.2%} ({cm[1,0]}/{support[1]} Non-MF patients misclassified)")
    
    # Show misclassified patients
    misclassified = [r for r in results if not r['correct']]
    if misclassified:
        print(f"\n❌ Misclassified Patients ({len(misclassified)}):")
        for r in misclassified:
            print(f"  • {r['patient']}")
            print(f"      True: {r['true_label']} | Predicted: {r['predicted_label']} | "
                  f"Confidence: {r['confidence']:.2%} | Images: {r['num_images']}")

# ============================================================================
# 5. Comparison Summary
# ============================================================================

print(f"\n\n{'='*80}")
print("COMPARISON OF AGGREGATION METHODS")
print(f"{'='*80}\n")

print(f"{'Method':<20} {'Accuracy':<12} {'F1-Score':<12} {'MF Recall':<12} {'AUC':<12}")
print("-"*80)

for method in aggregation_methods:
    res = all_method_results[method]
    acc = res['accuracy']
    f1 = res['f1_weighted']
    mf_recall = res['recall_per_class'][0]
    auc = res['auc'] if res['auc'] is not None else 0.0
    
    print(f"{method:<20} {acc:.4f}       {f1:.4f}       {mf_recall:.4f}       {auc:.4f}")

# ============================================================================
# 6. Final Recommendation
# ============================================================================

print(f"\n\n{'='*80}")
print("🎯 FINAL RECOMMENDATION FOR x20 MODEL")
print(f"{'='*80}\n")

# Find best aggregation method
best_method = max(aggregation_methods, 
                  key=lambda m: all_method_results[m]['f1_weighted'])
best_results = all_method_results[best_method]

print(f"Best Aggregation Method: {best_method.upper()}")
print(f"  Patient-Level Accuracy: {best_results['accuracy']:.2%}")
print(f"  MF Recall (Sensitivity): {best_results['recall_per_class'][0]:.2%}")
print(f"  Non-MF Recall (Specificity): {best_results['recall_per_class'][1]:.2%}")

# Decision criteria
mf_recall = best_results['recall_per_class'][0]
accuracy = best_results['accuracy']

print(f"\n💡 Clinical Decision:")

if mf_recall >= 0.85 and accuracy >= 0.80:
    print(f"  ✅ INCLUDE x20 in your ensemble")
    print(f"     → Performance is good enough for clinical support")
    print(f"     → Can complement x20 predictions")
    print(f"     → Use {best_method} aggregation for patient-level predictions")
elif mf_recall >= 0.75 and accuracy >= 0.70:
    print(f"  ⚠️  CONDITIONALLY INCLUDE x20")
    print(f"     → Performance is acceptable but not great")
    print(f"     → Use only as SECONDARY input to x20")
    print(f"     → Flag low-confidence predictions for review")
    print(f"     → Consider: x20 vote counts more (e.g., 70% x20, 30% x10)")
else:
    print(f"  ❌ DO NOT INCLUDE x20 in clinical deployment")
    print(f"     → Performance too unreliable for patient care")
    print(f"     → Missing too many MF cases ({100-mf_recall*100:.1f}%)")
    print(f"     → Use only x20 model for now")
    print(f"     → Collect more x20 data before reconsidering")

# Comparison with CV results
print(f"\n📊 Comparison with 5-Fold CV Patch-Level Results:")
print(f"  CV Mean Accuracy: 73.8% (patch-level)")
print(f"  Patient-Level Accuracy: {accuracy:.1%} (patient-level)")
print(f"  ")
if accuracy > 0.738:
    print(f"  ✓ Patient-level aggregation improves performance!")
    print(f"    → Multiple images per patient provides redundancy")
else:
    print(f"  → Similar or worse than patch-level CV")
    print(f"    → Model struggles even with multiple images")

print(f"\n{'='*80}\n")

# Save detailed results for further analysis
import json
output_path = Path("G:/My Drive/CLPD-MF-Dataset/Local Models/Hazem/CV_Results/x20_patient_level_test.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_data = {
    'model_path': str(model_path),
    'aggregation_methods': {
        method: {
            'accuracy': float(res['accuracy']),
            'f1_weighted': float(res['f1_weighted']),
            'mf_recall': float(res['recall_per_class'][0]),
            'nonmf_recall': float(res['recall_per_class'][1]),
            'confusion_matrix': res['confusion_matrix'].tolist(),
            'auc': float(res['auc']) if res['auc'] is not None else None
        }
        for method, res in all_method_results.items()
    },
    'best_method': best_method,
    'recommendation': 'include' if mf_recall >= 0.85 and accuracy >= 0.80 else 
                     'conditional' if mf_recall >= 0.75 and accuracy >= 0.70 else 'exclude'
}

with open(output_path, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"✓ Detailed results saved to: {output_path}")

Loading model from: C:\Users\Mohamed Hazem\Graduation Project\Dr. Rushdy\CLPD Dr. Kariman\MF Clasasifier\Trained Models\model_resnet50_x20.pth


C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_12252\2071520577.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=de

✓ Model loaded successfully

TESTING x20 MODEL WITH DIFFERENT AGGREGATION STRATEGIES

PATIENT-LEVEL TESTING FOR x10 MODEL
Aggregation method: mean

Total x10 patients: 48
  MF patients: 17
  Non-MF patients: 31


RESULTS - Aggregation Method: MEAN

📊 Overall Metrics:
  Accuracy: 0.9583 (46/48 patients correct)
  F1-Score (weighted): 0.9588
  AUC: 1.0000

🎯 Per-Class Metrics:
  MF:
    Precision: 0.8947
    Recall: 1.0000
    F1-Score: 0.9444
    Support: 17 patients
  Non-MF:
    Precision: 1.0000
    Recall: 0.9355
    F1-Score: 0.9667
    Support: 31 patients

📋 Confusion Matrix:
                Predicted MF  Predicted Non-MF
  Actual MF              17                    0
  Actual Non-MF           2                   29

  ⚠️  False Negative Rate: 0.00% (0/17 MF patients missed)
  ⚠️  False Positive Rate: 6.45% (2/31 Non-MF patients misclassified)

❌ Misclassified Patients (2):
  • omar 179-10-21
      True: Non-MF | Predicted: MF | Confidence: 54.64% | Images: 9
  • zyad 53-12-21


In [ ]:
# Patient-Level Testing Cell for x20 Model EfficientNetB2
import torch
import numpy as np
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)
import torch.nn.functional as F

device = torch.device('cuda')


# ============================================================================
# 1. Load the Previously Trained x20 Model
# ============================================================================

def load_pretrained_model(model_path, model_name='efficientnet_b2', num_classes=2):
    """Load a previously trained model from disk"""
    print(f"Loading model from: {model_path}")
    
    model = create_model(
        model_name=model_name,
        pretrained=False,  # Don't load ImageNet weights
        num_classes=num_classes,
        dropout=0.3
    )
    
    # Load saved weights
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    print(f"✓ Model loaded successfully")
    return model

# Load your previously trained x20 model
model_path = Path(r"C:/Users/Mohamed Hazem/Graduation Project/Dr. Rushdy/CLPD Dr. Kariman/MF Clasasifier/Trained Models/model_tf_efficientnet_b2_x20.pth")
if not model_path.exists():
    raise FileNotFoundError(f"Model not found at: {model_path}")

x20_model = load_pretrained_model(
    model_path, 
    model_name='tf_efficientnet_b2',  # Your previous model was tf_efficientnet_b2
    num_classes=2
)

print("="*80)

# ============================================================================
# 2. Patient-Level Prediction Function
# ============================================================================

def predict_patient(patient_rows, model, device, transforms=val_tf, aggregation='mean'):
    """
    Predict diagnosis for one patient by aggregating predictions across all their images.
    
    Args:
        patient_rows: List of image rows for this patient
        model: Trained model
        device: torch device
        transforms: Image transforms
        aggregation: 'mean', 'majority', or 'max_confidence'
    
    Returns:
        dict with prediction, confidence, and per-image details
    """
    model.eval()
    
    all_probs = []
    all_preds = []
    image_details = []
    
    with torch.no_grad():
        for row in patient_rows:
            # Extract patches from this image
            patches = extract_and_cache_patches(
                row['path'], 
                patch_size=512, 
                stride=256, 
                max_patches_per_image=100
            )
            
            if len(patches) == 0:
                continue
            
            # Predict on all patches from this image
            patch_probs = []
            for patch_path in patches:
                img = Image.open(patch_path).convert('RGB')
                img_tensor = transforms(img).unsqueeze(0).to(device)
                
                outputs = model(img_tensor)
                probs = F.softmax(outputs, dim=1).cpu().numpy()[0]
                patch_probs.append(probs)
            
            # Average predictions across patches for this image
            image_prob = np.mean(patch_probs, axis=0)
            image_pred = image_prob.argmax()
            
            all_probs.append(image_prob)
            all_preds.append(image_pred)
            
            image_details.append({
                'image_name': row['path'].name,
                'num_patches': len(patches),
                'prob_MF': image_prob[0],
                'prob_NonMF': image_prob[1],
                'prediction': 'MF' if image_pred == 0 else 'Non-MF'
            })
    
    if len(all_probs) == 0:
        return None
    
    # Aggregate across all images for patient-level prediction
    if aggregation == 'mean':
        # Average probabilities across all images
        patient_prob = np.mean(all_probs, axis=0)
        patient_pred = patient_prob.argmax()
        confidence = patient_prob[patient_pred]
        
    elif aggregation == 'majority':
        # Majority vote of image predictions
        vote_counts = Counter(all_preds)
        patient_pred = vote_counts.most_common(1)[0][0]
        confidence = vote_counts[patient_pred] / len(all_preds)
        patient_prob = np.mean(all_probs, axis=0)  # Still compute for reference
        
    elif aggregation == 'max_confidence':
        # Take prediction from most confident image
        confidences = [max(prob) for prob in all_probs]
        max_idx = np.argmax(confidences)
        patient_prob = all_probs[max_idx]
        patient_pred = patient_prob.argmax()
        confidence = patient_prob[patient_pred]
    
    return {
        'prediction': patient_pred,
        'predicted_class': 'MF' if patient_pred == 0 else 'Non-MF',
        'confidence': float(confidence),
        'prob_MF': float(patient_prob[0]),
        'prob_NonMF': float(patient_prob[1]),
        'num_images': len(patient_rows),
        'num_predictions': len(all_preds),
        'image_details': image_details,
        'aggregation_method': aggregation
    }

# ============================================================================
# 3. Test on ALL Patients at x20
# ============================================================================

def test_all_patients_x20(all_images, model, mag='x20', aggregation='mean'):
    """
    Test model on all patients with x20 images.
    Returns patient-level predictions and metrics.
    """
    print(f"\n{'='*80}")
    print(f"PATIENT-LEVEL TESTING FOR x20 MODEL")
    print(f"Aggregation method: {aggregation}")
    print(f"{'='*80}\n")
    
    # Group images by patient
    x20_images = [r for r in all_images if r['mag'] == mag]
    
    # Organize by patient
    patient_groups = defaultdict(list)
    for row in x20_images:
        patient_groups[row['patient']].append(row)
    
    print(f"Total x20 patients: {len(patient_groups)}")
    
    # Get ground truth labels for each patient
    patient_true_labels = {}
    for patient, rows in patient_groups.items():
        # All images from same patient should have same label
        labels = set(r['label'] for r in rows)
        if len(labels) > 1:
            print(f"⚠️  Warning: Patient {patient} has mixed labels: {labels}")
        patient_true_labels[patient] = list(labels)[0]
    
    # Count patients per class
    mf_patients = sum(1 for label in patient_true_labels.values() if label == 'MF')
    nonmf_patients = len(patient_true_labels) - mf_patients
    print(f"  MF patients: {mf_patients}")
    print(f"  Non-MF patients: {nonmf_patients}")
    print()
    
    # Predict for each patient
    results = []
    y_true = []
    y_pred = []
    y_probs = []
    
    label_map = {'MF': 0, 'Non-MF': 1}
    
    for patient, rows in patient_groups.items():
        true_label = patient_true_labels[patient]
        true_label_idx = label_map[true_label]
        
        # Get patient-level prediction
        pred_result = predict_patient(rows, model, device, aggregation=aggregation)
        
        if pred_result is None:
            print(f"⚠️  Skipping {patient} - no valid patches")
            continue
        
        pred_label_idx = pred_result['prediction']
        
        # Store for metrics
        y_true.append(true_label_idx)
        y_pred.append(pred_label_idx)
        y_probs.append([pred_result['prob_MF'], pred_result['prob_NonMF']])
        
        # Store detailed results
        results.append({
            'patient': patient,
            'true_label': true_label,
            'predicted_label': pred_result['predicted_class'],
            'correct': true_label == pred_result['predicted_class'],
            'confidence': pred_result['confidence'],
            'prob_MF': pred_result['prob_MF'],
            'prob_NonMF': pred_result['prob_NonMF'],
            'num_images': pred_result['num_images'],
            'image_details': pred_result['image_details']
        })
    
    return results, y_true, y_pred, y_probs

# ============================================================================
# 4. Run Tests with Different Aggregation Methods
# ============================================================================

print("\n" + "="*80)
print("TESTING x20 MODEL WITH DIFFERENT AGGREGATION STRATEGIES")
print("="*80)

aggregation_methods = ['mean', 'majority', 'max_confidence']
all_method_results = {}

for agg_method in aggregation_methods:
    results, y_true, y_pred, y_probs = test_all_patients_x20(
        all_images, 
        x20_model, 
        mag='x20', 
        aggregation=agg_method
    )
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=[0, 1], zero_division=0
    )
    
    # Weighted metrics
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    # AUC
    try:
        y_probs_array = np.array(y_probs)
        auc = roc_auc_score(y_true, y_probs_array[:, 1])
    except:
        auc = None
    
    # Store results
    all_method_results[agg_method] = {
        'accuracy': accuracy,
        'precision_per_class': precision,
        'recall_per_class': recall,
        'f1_per_class': f1,
        'support': support,
        'precision_weighted': precision_w,
        'recall_weighted': recall_w,
        'f1_weighted': f1_w,
        'confusion_matrix': cm,
        'auc': auc,
        'results': results
    }
    
    # Print results
    print(f"\n{'='*80}")
    print(f"RESULTS - Aggregation Method: {agg_method.upper()}")
    print(f"{'='*80}")
    
    print(f"\n📊 Overall Metrics:")
    print(f"  Accuracy: {accuracy:.4f} ({int(accuracy*len(y_true))}/{len(y_true)} patients correct)")
    print(f"  F1-Score (weighted): {f1_w:.4f}")
    if auc is not None:
        print(f"  AUC: {auc:.4f}")
    
    print(f"\n🎯 Per-Class Metrics:")
    class_names = ['MF', 'Non-MF']
    for i, cls in enumerate(class_names):
        print(f"  {cls}:")
        print(f"    Precision: {precision[i]:.4f}")
        print(f"    Recall: {recall[i]:.4f}")
        print(f"    F1-Score: {f1[i]:.4f}")
        print(f"    Support: {support[i]} patients")
    
    print(f"\n📋 Confusion Matrix:")
    print(f"                Predicted MF  Predicted Non-MF")
    print(f"  Actual MF          {cm[0,0]:6d}         {cm[0,1]:12d}")
    print(f"  Actual Non-MF      {cm[1,0]:6d}         {cm[1,1]:12d}")
    
    # Calculate error rates
    if support[0] > 0:
        fnr = cm[0,1] / support[0]  # False negative rate
        print(f"\n  ⚠️  False Negative Rate: {fnr:.2%} ({cm[0,1]}/{support[0]} MF patients missed)")
    
    if support[1] > 0:
        fpr = cm[1,0] / support[1]  # False positive rate
        print(f"  ⚠️  False Positive Rate: {fpr:.2%} ({cm[1,0]}/{support[1]} Non-MF patients misclassified)")
    
    # Show misclassified patients
    misclassified = [r for r in results if not r['correct']]
    if misclassified:
        print(f"\n❌ Misclassified Patients ({len(misclassified)}):")
        for r in misclassified:
            print(f"  • {r['patient']}")
            print(f"      True: {r['true_label']} | Predicted: {r['predicted_label']} | "
                  f"Confidence: {r['confidence']:.2%} | Images: {r['num_images']}")

# ============================================================================
# 5. Comparison Summary
# ============================================================================

print(f"\n\n{'='*80}")
print("COMPARISON OF AGGREGATION METHODS")
print(f"{'='*80}\n")

print(f"{'Method':<20} {'Accuracy':<12} {'F1-Score':<12} {'MF Recall':<12} {'AUC':<12}")
print("-"*80)

for method in aggregation_methods:
    res = all_method_results[method]
    acc = res['accuracy']
    f1 = res['f1_weighted']
    mf_recall = res['recall_per_class'][0]
    auc = res['auc'] if res['auc'] is not None else 0.0
    
    print(f"{method:<20} {acc:.4f}       {f1:.4f}       {mf_recall:.4f}       {auc:.4f}")

# ============================================================================
# 6. Final Recommendation
# ============================================================================

print(f"\n\n{'='*80}")
print("🎯 FINAL RECOMMENDATION FOR x20 MODEL")
print(f"{'='*80}\n")

# Find best aggregation method
best_method = max(aggregation_methods, 
                  key=lambda m: all_method_results[m]['f1_weighted'])
best_results = all_method_results[best_method]

print(f"Best Aggregation Method: {best_method.upper()}")
print(f"  Patient-Level Accuracy: {best_results['accuracy']:.2%}")
print(f"  MF Recall (Sensitivity): {best_results['recall_per_class'][0]:.2%}")
print(f"  Non-MF Recall (Specificity): {best_results['recall_per_class'][1]:.2%}")

# Decision criteria
mf_recall = best_results['recall_per_class'][0]
accuracy = best_results['accuracy']

print(f"\n💡 Clinical Decision:")

if mf_recall >= 0.85 and accuracy >= 0.80:
    print(f"  ✅ INCLUDE x20 in your ensemble")
    print(f"     → Performance is good enough for clinical support")
    print(f"     → Can complement x20 predictions")
    print(f"     → Use {best_method} aggregation for patient-level predictions")
elif mf_recall >= 0.75 and accuracy >= 0.70:
    print(f"  ⚠️  CONDITIONALLY INCLUDE x20")
    print(f"     → Performance is acceptable but not great")
    print(f"     → Use only as SECONDARY input to x20")
    print(f"     → Flag low-confidence predictions for review")
    print(f"     → Consider: x20 vote counts more (e.g., 70% x20, 30% x10)")
else:
    print(f"  ❌ DO NOT INCLUDE x20 in clinical deployment")
    print(f"     → Performance too unreliable for patient care")
    print(f"     → Missing too many MF cases ({100-mf_recall*100:.1f}%)")
    print(f"     → Use only x20 model for now")
    print(f"     → Collect more x20 data before reconsidering")

# Comparison with CV results
print(f"\n📊 Comparison with 5-Fold CV Patch-Level Results:")
print(f"  CV Mean Accuracy: 73.8% (patch-level)")
print(f"  Patient-Level Accuracy: {accuracy:.1%} (patient-level)")
print(f"  ")
if accuracy > 0.738:
    print(f"  ✓ Patient-level aggregation improves performance!")
    print(f"    → Multiple images per patient provides redundancy")
else:
    print(f"  → Similar or worse than patch-level CV")
    print(f"    → Model struggles even with multiple images")

print(f"\n{'='*80}\n")

# Save detailed results for further analysis
import json
output_path = Path("G:/My Drive/CLPD-MF-Dataset/Local Models/Hazem/CV_Results/x20_patient_level_test.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_data = {
    'model_path': str(model_path),
    'aggregation_methods': {
        method: {
            'accuracy': float(res['accuracy']),
            'f1_weighted': float(res['f1_weighted']),
            'mf_recall': float(res['recall_per_class'][0]),
            'nonmf_recall': float(res['recall_per_class'][1]),
            'confusion_matrix': res['confusion_matrix'].tolist(),
            'auc': float(res['auc']) if res['auc'] is not None else None
        }
        for method, res in all_method_results.items()
    },
    'best_method': best_method,
    'recommendation': 'include' if mf_recall >= 0.85 and accuracy >= 0.80 else 
                     'conditional' if mf_recall >= 0.75 and accuracy >= 0.70 else 'exclude'
}

with open(output_path, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"✓ Detailed results saved to: {output_path}")

Loading model from: C:\Users\Mohamed Hazem\Graduation Project\Dr. Rushdy\CLPD Dr. Kariman\MF Clasasifier\Trained Models\model_tf_efficientnet_b2_x20.pth
✓ Model loaded successfully

TESTING x20 MODEL WITH DIFFERENT AGGREGATION STRATEGIES

PATIENT-LEVEL TESTING FOR x20 MODEL
Aggregation method: mean

Total x20 patients: 55
  MF patients: 17
  Non-MF patients: 38



C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_24564\3733864524.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=de


RESULTS - Aggregation Method: MEAN

📊 Overall Metrics:
  Accuracy: 0.9636 (53/55 patients correct)
  F1-Score (weighted): 0.9642
  AUC: 1.0000

🎯 Per-Class Metrics:
  MF:
    Precision: 0.8947
    Recall: 1.0000
    F1-Score: 0.9444
    Support: 17 patients
  Non-MF:
    Precision: 1.0000
    Recall: 0.9474
    F1-Score: 0.9730
    Support: 38 patients

📋 Confusion Matrix:
                Predicted MF  Predicted Non-MF
  Actual MF              17                    0
  Actual Non-MF           2                   36

  ⚠️  False Negative Rate: 0.00% (0/17 MF patients missed)
  ⚠️  False Positive Rate: 5.26% (2/38 Non-MF patients misclassified)

❌ Misclassified Patients (2):
  • nour 29-30-4-24
      True: Non-MF | Predicted: MF | Confidence: 62.76% | Images: 25
  • omar 179-10-21
      True: Non-MF | Predicted: MF | Confidence: 61.36% | Images: 9

PATIENT-LEVEL TESTING FOR x20 MODEL
Aggregation method: majority

Total x20 patients: 55
  MF patients: 17
  Non-MF patients: 38


RESULTS -